In [2]:
!curl -L -o restaurant_inspections.csv "https://data.cityofnewyork.us/api/views/43nn-pn8j/rows.csv?accessType=DOWNLOAD"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  138M    0  138M    0     0  5026k      0 --:--:--  0:00:28 --:--:-- 6598k


In [4]:
import pandas as pd

print(df_recent.shape)

(288602, 27)


In [5]:
establishment_features = df_recent.groupby('CAMIS').agg(
    dba=('DBA', 'first'),
    boro=('BORO', 'first'),
    cuisine=('CUISINE DESCRIPTION', 'first'),
    total_inspections=('CAMIS', 'count'),
    total_critical=('CRITICAL FLAG', lambda x: (x == 'Critical').sum()),
    avg_score=('SCORE', 'mean'),
    latest_inspection=('INSPECTION DATE', 'max'),
    first_inspection=('INSPECTION DATE', 'min')
).reset_index()

establishment_features['critical_rate'] = (
    establishment_features['total_critical'] / establishment_features['total_inspections']
)

print(establishment_features.shape)
establishment_features.head()

(27301, 10)


,CAMIS,dba,boro,cuisine,total_inspections,total_critical,avg_score,latest_inspection,first_inspection,critical_rate
0,30075445,MORRIS PARK BAKE SHOP,Bronx,Bakery Products/Desserts,11,5,17.636364,2026-02-27,2023-08-01,0.454545
1,30191841,D.J. REYNOLDS,Manhattan,Irish,10,6,18.400000,2025-02-20,2023-04-23,0.600000
2,40356018,RIVIERA CATERERS,Brooklyn,American,4,1,6.666667,2025-09-17,2024-04-16,0.250000
3,40356483,WILKEN'S FINE FOOD,Brooklyn,Sandwiches,18,9,25.312500,2026-06-10,2023-11-16,0.500000
4,40356731,TASTE THE TROPICS ICE CREAM,Brooklyn,Frozen Desserts,8,2,11.125000,2026-06-25,2024-04-08,0.250000


In [6]:
temp_keywords = 'Cold TCS|Hot TCS|held above|held at or above|temperature'

df_recent['is_temp_violation'] = df_recent['VIOLATION DESCRIPTION'].str.contains(
    temp_keywords, case=False, na=False
)

temp_counts = df_recent[df_recent['is_temp_violation']].groupby('CAMIS').size().rename('temp_violation_count')

establishment_features = establishment_features.merge(temp_counts, on='CAMIS', how='left')
establishment_features['temp_violation_count'] = establishment_features['temp_violation_count'].fillna(0)

print(establishment_features['temp_violation_count'].describe())

count    27301.000000
mean         1.496575
std          1.767896
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max         17.000000
Name: temp_violation_count, dtype: float64


In [7]:
yearly_critical = (
    df_recent[df_recent['CRITICAL FLAG']=='Critical']
    .assign(year=df_recent['INSPECTION DATE'].dt.year)
    .groupby(['CAMIS','year']).size().unstack(fill_value=0)
)

establishment_features = establishment_features.merge(
    yearly_critical[[2024, 2025]] if 2024 in yearly_critical.columns and 2025 in yearly_critical.columns else yearly_critical,
    on='CAMIS', how='left'
)

In [9]:
print(yearly_critical.columns.tolist())

[2022, 2023, 2024, 2025, 2026]


In [10]:
yearly_critical = yearly_critical.reset_index()

establishment_features = establishment_features.merge(
    yearly_critical[['CAMIS', 2024, 2025]],
    on='CAMIS', how='left'
).fillna({2024: 0, 2025: 0})

establishment_features['critical_change_24_25'] = (
    establishment_features[2025] - establishment_features[2024]
)

print(establishment_features.shape)
print(establishment_features['critical_change_24_25'].describe())

(27301, 18)
count    27301.000000
mean         0.190542
std          3.168554
min        -21.000000
25%         -1.000000
50%          0.000000
75%          1.000000
max         26.000000
Name: critical_change_24_25, dtype: float64


In [12]:
import os
os.makedirs("/content/drive/MyDrive/restaurant_project", exist_ok=True)

establishment_features.to_csv("/content/drive/MyDrive/restaurant_project/establishment_features.csv", index=False)
print("Saved:", establishment_features.shape)

Saved: (27301, 18)
